This notebook builds the training data: for each collection, it saves a table listing every usable field with its crop type, its train/validation/test split (taken from FTW's chips), and its AEF and OlmoEarth embeddings.

In [ ]:
import json
import fsspec
import shapely
import numpy as np
import pandas as pd
import pyarrow as pa
import geopandas as gpd
from pathlib import Path
import pyarrow.compute as pc
import pyarrow.parquet as pq
from urllib.parse import urljoin
from IPython.display import display
pd.set_option('display.max_rows',20)

In [ ]:
CATALOGURL  = 'https://data.source.coop/ftw/benchmark-data/catalog.json'
RAWDIR      = Path('../data/raw')
INTERIMDIR  = Path('../data/interim')
PROCDIR     = Path('../data/processed')
SETTINGS    = json.loads((INTERIMDIR/'settings.json').read_text())
COLLECTIONS = pd.read_csv(INTERIMDIR/'collections.csv').collection.tolist()
MINFIELDS   = SETTINGS['minfields']
FILECOLUMNS = SETTINGS['filecolumns']
MAXFIELDS   = 200000
MINPIXELS   = 5
DROPCROPS   = []
SEED        = 42
REMOTE      = fsspec.filesystem('https')

In [ ]:
def build_file_url(collection,filetype):
    return urljoin(CATALOGURL,f"{collection}/{collection.split('/')[1]}_{filetype}.parquet")

def build_raw_path(collection,filetype):
    return RAWDIR/f"{collection.split('/')[1]}_{filetype}.parquet"

def save_raw_copy(collection,filetype):
    rawpath = build_raw_path(collection,filetype)
    if not rawpath.exists():
        partpath = rawpath.with_suffix('.part')
        with REMOTE.open(build_file_url(collection,filetype)) as file:
            parquet = pq.ParquetFile(file)
            schema  = pa.schema([parquet.schema_arrow.field(col) for col in FILECOLUMNS[filetype]],metadata=parquet.schema_arrow.metadata)
            with pq.ParquetWriter(partpath,schema,compression='zstd') as writer:
                for batch in parquet.iter_batches(batch_size=100000,columns=FILECOLUMNS[filetype]):
                    writer.write_batch(batch)
        partpath.rename(rawpath)

def scan_rows(rawpath,cols,ids):
    keep = pa.array(ids)
    for batch in pq.ParquetFile(rawpath).iter_batches(batch_size=50000,columns=cols):
        yield batch.filter(pc.is_in(batch['id'],value_set=keep))

def drop_fields(fields,keep,reason,log):
    log[reason] = int((~keep).sum())
    return fields[keep]

In [ ]:
def build_table(collection):
    for filetype in FILECOLUMNS:
        save_raw_copy(collection,filetype)
    log = {'collection':collection}

    labels = pd.read_parquet(build_raw_path(collection,'fields'),columns=['id','hcat:name'],dtype_backend='pyarrow')
    pixels = pd.read_parquet(build_raw_path(collection,'field_embeddings'),columns=['id','pixels','pixel_rule'],dtype_backend='pyarrow')
    fields = labels.merge(pixels.drop_duplicates('id'),on='id',how='left').rename(columns={'hcat:name':'crop'})
    log['start'] = len(fields)
    fields = drop_fields(fields,fields.crop.notna(),'nolabel',log)
    fields = drop_fields(fields,~fields.id.duplicated(keep=False),'duplicateid',log)
    fields = drop_fields(fields,(fields.pixels>=MINPIXELS)&(fields.pixel_rule!='touched'),'small',log)
    fields = drop_fields(fields,~fields.crop.isin(DROPCROPS),'dropcrops',log)
    counts = fields.crop.value_counts()
    fields = drop_fields(fields,fields.crop.map(counts)>=MINFIELDS,'rarecrop',log)
    quota  = (fields.crop.value_counts()*min(1,MAXFIELDS/len(fields))).round().clip(lower=MINFIELDS)
    fields = fields.sample(frac=1,random_state=SEED)
    fields = drop_fields(fields,fields.groupby('crop').cumcount()<fields.crop.map(quota),'capped',log)[['id','crop']]
    ids    = fields.id.to_numpy()

    stats      = ['aef_mean','aef_std','olmoearth_mean','olmoearth_std']
    batches    = [(batch['id'].to_numpy(zero_copy_only=False),np.hstack([batch[stat].flatten().to_numpy().reshape(batch.num_rows,-1) for stat in stats]))
                  for batch in scan_rows(build_raw_path(collection,'field_embeddings'),['id']+stats,ids)]
    featureids = np.concatenate([batchids for batchids,_ in batches])
    features   = np.vstack([values for _,values in batches])
    del batches

    chips     = gpd.read_parquet(build_raw_path(collection,'chips'))
    chipgeoms = np.asarray(chips.geometry)
    locations = []
    for batch in scan_rows(build_raw_path(collection,'fields'),['id','geometry'],ids):
        geoms     = shapely.make_valid(shapely.from_wkb(batch['geometry'].to_numpy(zero_copy_only=False)))
        fieldidx,chipidx = chips.sindex.query(geoms,predicate='intersects')
        overlap   = shapely.area(shapely.intersection(geoms[fieldidx],chipgeoms[chipidx]))
        best      = pd.Series(chipidx,index=fieldidx).iloc[np.argsort(overlap)].groupby(level=0).last()
        centroids = shapely.centroid(geoms)
        locations.append(pd.DataFrame({'id':batch['id'].to_numpy(zero_copy_only=False),'chipidx':best.reindex(range(len(geoms))).to_numpy(),
                                       'lon':shapely.get_x(centroids),'lat':shapely.get_y(centroids)}))
    location = pd.concat(locations)

    info   = pd.DataFrame({'id':featureids}).merge(fields.astype(object),on='id',how='left').merge(location,on='id',how='left')
    inchip = info.chipidx.notna().to_numpy()
    log['outsidechips'] = int((~inchip).sum())
    info   = info[inchip].reset_index(drop=True)
    chipidx = info.pop('chipidx').astype(int).to_numpy()
    info.insert(2,'chip',chips.id.to_numpy()[chipidx])
    info.insert(3,'split',chips.split.to_numpy()[chipidx])
    names  = [f'{model}{i:03d}' for model in ('aef','olmoearth') for i in range(128)]
    table  = pd.concat([info,pd.DataFrame(features[inchip],columns=names)],axis=1)
    table.to_parquet(PROCDIR/f"fields_{collection.split('/')[1]}.parquet",index=False)

    log.update(kept=len(table),crops=table.crop.nunique(),**table.split.value_counts().reindex(['train','val','test']).to_dict())
    print(f"{collection}: kept {len(table):,} of {log['start']:,} fields, {log['crops']} crops")
    return log

In [ ]:
todo = [collection for collection in COLLECTIONS if not (PROCDIR/f"fields_{collection.split('/')[1]}.parquet").exists()]
print(f'Building {len(todo)} of {len(COLLECTIONS)} collections (the rest are already saved)...')
logs = [build_table(collection) for collection in todo]

logpath = INTERIMDIR/'build_log.csv'
log     = pd.concat([pd.read_csv(logpath) if logpath.exists() else None,pd.DataFrame(logs)]).drop_duplicates('collection',keep='last')
log     = log.set_index('collection').loc[[collection for collection in COLLECTIONS if collection in set(log.collection)]].reset_index()
log.to_csv(logpath,index=False)

In [ ]:
headers = {'collection':'Collection','start':'Fields','nolabel':'No Label','duplicateid':'Duplicate ID','small':f'< {MINPIXELS} Pixels',
           'dropcrops':'In DROPCROPS','rarecrop':f'Crop < {MINFIELDS} Fields','capped':'Capped','outsidechips':'Outside Chips',
           'kept':'Kept','crops':'Crops','train':'Train','val':'Val','test':'Test'}

print(f"{log.kept.sum():,} fields kept across {len(log)} collections; the middle columns count the fields dropped at each step")
display(log[list(headers)].rename(columns=headers).style.format('{:,}',subset=list(headers.values())[1:]).hide(axis='index'))